# Word count with PySpark

#### Topics covered in this example
* Write a file to HDFS, read the file and perform word count on the data.

***

## Prerequisites
<div class="alert alert-block alert-info">
<b>NOTE :</b> In order to execute this notebook successfully as is, please ensure the following prerequisites are completed.</div>

* The EMR cluster attached to this notebook should have the `Spark` application installed.
* This notebook uses the `PySpark` kernel.
***

## Introduction
In this example we write a file to hdfs, use pyspark to count the occurrence of each word in the file stored in hdfs and store the results to s3.
***

## Setup
1. Create a S3 bucket to save your results or use an existing s3 bucket. For example: `s3://EXAMPLE-BUCKET/word-count/`
***

## Example

Create a test data frame with some sample records.
We will use the `createDataFrame()` method to create and `printSchema()` method to print out the schema.

In [3]:
wordsDF = sqlContext.createDataFrame([("emr",), ("spark",), ("example",), ("spark",), ("pyspark",), ("python",),
             ("example",), ("emr",), ("example",), ("spark",), ("pyspark",), ("python",)], ["words"])
wordsDF.show()
wordsDF.printSchema()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-------+
|  words|
+-------+
|    emr|
|  spark|
|example|
|  spark|
|pyspark|
| python|
|example|
|    emr|
|example|
|  spark|
|pyspark|
| python|
+-------+

root
 |-- words: string (nullable = true)

Print out the number of unique words so that we can verify this number with the end result count.

In [4]:
uniqueWordsCount = wordsDF.distinct().groupBy().count().head()[0]
print(uniqueWordsCount)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

5

This step only shows an example on how to write to hdfs.
You can use an existing file stored in hdfs and read it as shown in the next steps.

In [11]:
wordsDF.write.csv("hdfs:///user/hadoop/test-data_indra.csv")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Read the csv file from hdfs and store in RDD.

In [12]:
wordsData = sc.textFile("hdfs:///user/hadoop/test-data.csv")
wordsData.count()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

12

Display the contents of the file.

In [13]:
wordsData.collect()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

['emr', 'spark', 'example', 'spark', 'pyspark', 'python', 'example', 'emr', 'example', 'spark', 'pyspark', 'python']

Count the occurance of each word and print the count of the result. This should be equal to the number of unique words we found earlier.

In [14]:
wordsCounts = wordsData.flatMap(lambda line: line.split(" ")) \
                       .map(lambda word: (word, 1)) \
                       .reduceByKey(lambda a, b: a+b)
wordsCounts.count()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

5

Display the count for each word.

In [15]:
wordsCounts.collect()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

[('python', 2), ('pyspark', 2), ('spark', 3), ('emr', 2), ('example', 3)]

Save the results to your s3 bucket. The results are stored in the key `word-count` and split based on paritions.

In [17]:
wordsCounts.saveAsTextFile("s3://ihbucket/indrawp/tesindra") # Change this to the S3 location that you created in Setup step 1.

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
empData=sc.textFile("hdfs://user/hadoop/employee_data.csv")
empData.count()